# 03 -- Apriori Baseline
Association rule mining on Instacart order baskets, with leave-one-out evaluation against held-out test users.

In [1]:
# Install mlxtend if not already installed
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "mlxtend", "-q"], check=True)

CompletedProcess(args=['C:\\Users\\user\\AppData\\Local\\Programs\\Python\\Python312\\python.exe', '-m', 'pip', 'install', 'mlxtend', '-q'], returncode=0)

In [2]:
import numpy as np
import pandas as pd
import json
import gc
import zipfile
from collections import Counter
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

In [3]:
# Hyperparameters -- adjust here only
MIN_SUPPORT    = 0.01
MIN_CONFIDENCE = 0.2

# Paths
DATA_DIR = "../data/processed"
RAW_ZIP  = "../data/raw/instacart_dataset.zip"

In [4]:
# Load order data
orders = pd.read_csv(f"{DATA_DIR}/orders_subset.csv")
order_products = pd.read_csv(f"{DATA_DIR}/order_products_subset.csv")

# Filter to 'prior' orders only (training baskets -- not the test/eval split)
prior_orders = orders[orders["eval_set"] == "prior"][["order_id"]]

# Join to get product lists for each prior order
prior_order_products = prior_orders.merge(order_products, on="order_id")

print(f"Prior orders:    {prior_orders.shape[0]:,}")
print(f"Prior products:  {prior_order_products.shape[0]:,}")
print(prior_order_products.head())

Prior orders:    189,462
Prior products:  1,911,053
   order_id  product_id  add_to_cart_order  reordered
0   2539329         196                  1          0
1   2539329       14084                  2          0
2   2539329       12427                  3          0
3   2539329       26088                  4          0
4   2539329       26405                  5          0


In [5]:
# Group by order_id -> list of product_ids (as strings for mlxtend)
baskets = (
    prior_order_products
    .groupby("order_id")["product_id"]
    .apply(lambda x: [str(p) for p in x.tolist()])
    .tolist()
)

print(f"Total baskets: {len(baskets):,}")
print(f"Sample basket (first 5 items): {baskets[0][:5]}")

# Free memory from intermediate dataframes
del prior_order_products, prior_orders
gc.collect()

# Pre-filter: keep only products appearing in >= MIN_SUPPORT fraction of baskets
# (products below this threshold can never appear in frequent itemsets)
product_counter = Counter()
for basket in baskets:
    product_counter.update(basket)

min_count = int(MIN_SUPPORT * len(baskets))
frequent_products = {p for p, c in product_counter.items() if c >= min_count}
print(f"Products meeting min_support threshold: {len(frequent_products):,} (of {len(product_counter):,} total)")

# Filter baskets to only contain frequent products
baskets_filtered = [[p for p in basket if p in frequent_products] for basket in baskets]
baskets_filtered = [b for b in baskets_filtered if len(b) > 0]
print(f"Baskets after filtering: {len(baskets_filtered):,}")

del baskets
gc.collect()

Total baskets: 189,462
Sample basket (first 5 items): ['28204', '39108', '4472', '43950', '29228']


Products meeting min_support threshold: 105 (of 36,741 total)


Baskets after filtering: 140,033


0

In [6]:
# One-hot encode using TransactionEncoder (sparse=True for memory efficiency)
te = TransactionEncoder()
te_array = te.fit_transform(baskets_filtered, sparse=True)
basket_df = pd.DataFrame.sparse.from_spmatrix(te_array, columns=te.columns_)

print(f"Transaction matrix shape: {basket_df.shape}")
print(f"Sparse density: {te_array.nnz / (te_array.shape[0] * te_array.shape[1]):.4f}")

Transaction matrix shape: (140033, 105)
Sparse density: 0.0310


In [7]:
# Mine frequent itemsets
print(f"Mining with min_support={MIN_SUPPORT}, min_confidence={MIN_CONFIDENCE} ...")
frequent_itemsets = apriori(basket_df, min_support=MIN_SUPPORT, use_colnames=True, low_memory=True)
frequent_itemsets["length"] = frequent_itemsets["itemsets"].apply(len)

print(f"Frequent itemsets found: {len(frequent_itemsets):,}")
print(frequent_itemsets.sort_values("support", ascending=False).head(10))

# Free the large transaction matrix
del basket_df, te_array, baskets_filtered
gc.collect()

# Generate association rules
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=MIN_CONFIDENCE)
rules = rules.sort_values("confidence", ascending=False).reset_index(drop=True)

print(f"\nAssociation rules mined: {len(rules):,}")
print(rules[["antecedents", "consequents", "support", "confidence", "lift"]].head(10))

Mining with min_support=0.01, min_confidence=0.2 ...


Frequent itemsets found: 134
     support            itemsets  length
26  0.198246  frozenset({24852})       1
3   0.160919  frozenset({13176})       1
15  0.116930  frozenset({21137})       1
18  0.099591  frozenset({21903})       1
85  0.092307  frozenset({47209})       1
87  0.071683  frozenset({47766})       1
86  0.065556  frozenset({47626})       1
29  0.061721  frozenset({26209})       1
6   0.059050  frozenset({16797})       1
37  0.058536  frozenset({27966})       1

Association rules mined: 19
          antecedents         consequents   support  confidence      lift
0  frozenset({28204})  frozenset({24852})  0.013461    0.366304  1.847723
1  frozenset({45066})  frozenset({24852})  0.011590    0.359468  1.813243
2  frozenset({47766})  frozenset({24852})  0.022031    0.307332  1.550255
3   frozenset({4920})  frozenset({24852})  0.010219    0.299498  1.510737
4  frozenset({27966})  frozenset({13176})  0.017160    0.293156  1.821759
5  frozenset({16797})  frozenset({24852})  0.01

## Evaluation: Leave-One-Out Protocol

Using the 1,000 held-out test users from X_test.npz.

**Protocol:** For each test user whose last order has known products (eval_set='train' in Instacart):
- **Query** = all product IDs from their prior orders
- **Ground truth** = the product IDs in their held-out last order
- **Recommendations** = top-K rule consequents that fire on the query items
- **HR@K** = fraction of users with at least 1 hit in top-K

Note: Instacart's eval_set='test' orders don't have product labels (public leaderboard hold-out),
so only eval_set='train' users can be evaluated. We load the train order products from the raw zip.

In [8]:
# Load test user IDs
test_data = np.load(f"{DATA_DIR}/X_test.npz")
X_test = test_data["sequences"]   # shape: (1000, 100), int32, PAD=0
test_user_ids = test_data["user_ids"]  # shape: (1000,)

print(f"X_test shape: {X_test.shape}, dtype: {X_test.dtype}")
print(f"Test users: {len(test_user_ids)}")

# Load vocabulary: product_id (str) -> index (int)
with open(f"{DATA_DIR}/vocab.json", "r") as f:
    vocab = json.load(f)

# Build reverse vocab: index (int) -> product_id (str)
idx_to_product = {v: k for k, v in vocab.items()}

print(f"Vocab size: {len(vocab):,} entries (including PAD at index 0)")

X_test shape: (1000, 100), dtype: int32
Test users: 1000


Vocab size: 5,001 entries (including PAD at index 0)


In [9]:
# Load train order products from raw zip to get ground truth for eval_set='train' users
with zipfile.ZipFile(RAW_ZIP) as zf:
    train_products_raw = pd.read_csv(zf.open("order_products__train.csv"))

print(f"Raw train order products: {len(train_products_raw):,}")

# Get the train-eval orders for our test users
train_eval_orders = orders[
    (orders["user_id"].isin(test_user_ids)) &
    (orders["eval_set"] == "train")
][["order_id", "user_id"]]

print(f"Test users with eval_set='train' (evaluable): {len(train_eval_orders)}")

# Join to get products in each user's last order
last_order_products = train_eval_orders.merge(train_products_raw, on="order_id")
print(f"Ground truth product entries: {len(last_order_products):,}")

# Build ground truth map: user_id -> set of product_id strings
ground_truth_map = (
    last_order_products
    .groupby("user_id")["product_id"]
    .apply(lambda x: set(str(p) for p in x))
    .to_dict()
)

# Build query from prior orders for each test user
prior_user_orders = orders[
    (orders["user_id"].isin(test_user_ids)) &
    (orders["eval_set"] == "prior")
][["order_id", "user_id"]]

prior_user_products = prior_user_orders.merge(order_products, on="order_id")
query_map = (
    prior_user_products
    .groupby("user_id")["product_id"]
    .apply(lambda x: set(str(p) for p in x))
    .to_dict()
)

print(f"Users with ground truth: {len(ground_truth_map)}")
print(f"Users with prior query products: {len(query_map)}")

# Free raw train data
del train_products_raw
gc.collect()

# Sample
evaluable_uids = [uid for uid in test_user_ids if uid in ground_truth_map and uid in query_map]
print(f"\nEvaluable test users: {len(evaluable_uids)}")
if evaluable_uids:
    s = evaluable_uids[0]
    print(f"Sample user {s}: {len(query_map[s])} prior items, {len(ground_truth_map[s])} ground truth items")

Raw train order products: 1,384,617
Test users with eval_set='train' (evaluable): 640
Ground truth product entries: 7,074


Users with ground truth: 640
Users with prior query products: 1000

Evaluable test users: 640
Sample user 36489: 66 prior items, 24 ground truth items


In [10]:
def apriori_recommend(query_products, rules_df, top_k=10):
    """
    Given a set of product_id strings (query_products),
    find all rules whose antecedents are a subset of the query,
    collect consequent product IDs ranked by confidence desc,
    return top_k unique consequent IDs not already in query.

    Args:
        query_products: set of product_id strings (str)
        rules_df: DataFrame with 'antecedents', 'consequents', 'confidence' columns
        top_k: int, number of recommendations to return

    Returns:
        list of product_id strings (str), length <= top_k
    """
    query_set = set(str(p) for p in query_products)
    
    # Filter rules where antecedents is a subset of query
    firing_rules = rules_df[
        rules_df["antecedents"].apply(lambda ant: ant.issubset(query_set))
    ].sort_values("confidence", ascending=False)
    
    # Collect consequents in confidence order, exclude items already in query
    recommendations = []
    seen = set(query_set)
    for _, row in firing_rules.iterrows():
        for item in row["consequents"]:
            if item not in seen:
                recommendations.append(item)
                seen.add(item)
        if len(recommendations) >= top_k:
            break
    
    return recommendations[:top_k]

In [11]:
# Sanity check: use arbitrary top products from frequent_itemsets
top_product = list(frequent_itemsets.sort_values("support", ascending=False).iloc[0]["itemsets"])[0]
test_recs = apriori_recommend({top_product}, rules, top_k=5)
print(f"Sample recs for product {top_product}: {test_recs}")

Sample recs for product 24852: []


In [12]:
# Evaluation using order-level ground truth
hits_at_5  = 0
hits_at_10 = 0
precision_at_5_total = 0.0
mrr_total = 0.0
evaluated = 0

for uid in evaluable_uids:
    query = query_map[uid]
    ground_truth = ground_truth_map[uid]

    # Get recommendations
    recs_10 = apriori_recommend(query, rules, top_k=10)
    recs_5  = recs_10[:5]
    evaluated += 1

    # HR@5: at least 1 hit in top-5
    if any(r in ground_truth for r in recs_5):
        hits_at_5 += 1

    # HR@10: at least 1 hit in top-10
    if any(r in ground_truth for r in recs_10):
        hits_at_10 += 1

    # Precision@5 = hits in top-5 / 5
    hits_in_5 = sum(1 for r in recs_5 if r in ground_truth)
    precision_at_5_total += hits_in_5 / 5

    # MRR = 1 / rank of first hit (0 if no hit)
    for rank, rec in enumerate(recs_10, start=1):
        if rec in ground_truth:
            mrr_total += 1.0 / rank
            break

HR5         = hits_at_5 / evaluated
HR10        = hits_at_10 / evaluated
PRECISION5  = precision_at_5_total / evaluated
MRR         = mrr_total / evaluated

print("=" * 45)
print(f"{'Metric':<20} {'Value':>10}")
print("=" * 45)
print(f"{'HR@5':<20} {HR5:>10.4f}")
print(f"{'HR@10':<20} {HR10:>10.4f}")
print(f"{'Precision@5':<20} {PRECISION5:>10.4f}")
print(f"{'MRR':<20} {MRR:>10.4f}")
print("=" * 45)
print(f"\nEvaluated on {evaluated} test users")
print(f"(Users with eval_set='test' excluded -- Instacart provides no ground truth for those)")
print(f"Expected HR@5 ~ 0.33 per PRD")

Metric                    Value
HR@5                     0.0312
HR@10                    0.0312
Precision@5              0.0063
MRR                      0.0273

Evaluated on 640 test users
(Users with eval_set='test' excluded -- Instacart provides no ground truth for those)
Expected HR@5 ~ 0.33 per PRD


In [13]:
# Save results to CSV -- one row per metric, columns: metric, value
baseline_results = pd.DataFrame([
    {"metric": "HR@5",        "value": HR5},
    {"metric": "HR@10",       "value": HR10},
    {"metric": "Precision@5", "value": PRECISION5},
    {"metric": "MRR",         "value": MRR},
])

save_path = f"{DATA_DIR}/baseline_results.csv"
baseline_results.to_csv(save_path, index=False)

print(f"Saved: {save_path}")
print(baseline_results.to_string(index=False))
# Also save to shared results dir for Phase 5/7 comparison notebooks
RESULTS_DIR = f'{MODELS_DIR}/../results'
import os
os.makedirs(RESULTS_DIR, exist_ok=True)
results_save_path = f'{RESULTS_DIR}/baseline_results.csv'
baseline_results.to_csv(results_save_path, index=False)
print(f'Also saved to: {results_save_path}')


Saved: ../data/processed/baseline_results.csv
     metric    value
       HR@5 0.031250
      HR@10 0.031250
Precision@5 0.006250
        MRR 0.027344


## Apriori Baseline: Limitations and LSTM Improvements

While Apriori association rule mining provides a useful and interpretable baseline for product recommendation, it carries several structural limitations that make it a poor choice for personalized, real-time grocery recommendations. Each limitation is discussed below, along with how the LSTM-based model in this project addresses it.

### 1. Ignores Item Order

Apriori treats each transaction as an unordered set of items, completely discarding the sequence in which a user added products to their basket or across multiple shopping sessions. This means it cannot learn that someone who buys coffee and a coffee maker is likely next to buy filters -- it only sees co-occurrence, not causality or progression. The resulting rules reflect population-level buying patterns rather than any individual's purchasing journey. The LSTM model, by contrast, processes item sequences in order, allowing it to capture directional purchasing patterns and predict what a user is likely to buy *next* given their recent history.

### 2. Ignores Temporal Context

Association rules mined from historical baskets carry no information about *when* a purchase is likely to happen. Shoppers behave differently on a Monday morning versus a Sunday evening -- cereal and milk spike in morning hours, while snacks and beverages peak on weekends -- but Apriori cannot incorporate this signal. Rules that fire in all contexts necessarily sacrifice precision for generality. The Time-Aware LSTM in Phase 5 addresses this by embedding hour-of-day, day-of-week, and days-since-last-order directly into the model, enabling recommendations that shift naturally with the user's current shopping context.

### 3. No Personalization

Apriori produces a single global ruleset applied identically to every user. A seasoned home cook and a college student buying instant noodles receive the same recommendations as long as their current basket looks similar. This means the model can never surface the genuinely personal preferences that differentiate individual shoppers. The LSTM model is trained on each user's own order history and encodes user-specific sequential patterns into its hidden state, producing recommendations that are meaningfully personalized to individuals rather than averaged across the population.

### 4. Cold Start Failure

When a user has no prior order history, Apriori has nothing to query against -- its rules require at least some items in the current basket to fire, and even then they reflect population averages rather than the new user's preferences. This is a critical failure mode for any production-grade recommendation system. FreshCart's three-tier cold start system (Phase 6) directly solves this: Tier 1 users (zero orders) receive globally popular items, Tier 2 users (1-2 orders) receive aisle-affinity recommendations, and only Tier 3 users (3+ orders) get personalized LSTM predictions -- ensuring no user ever sees an empty recommendation panel.

### 5. No Session Modeling

Apriori operates on completed, static order baskets -- it cannot model the dynamics of an *ongoing* shopping session where items are added incrementally and recommendations should update in real time. A rule mined from completed historical checkouts cannot reason about an in-progress cart where only 3 items have been added so far. The LSTM, combined with the live `/api/recommend` endpoint in Phase 8, re-scores recommendations after every cart change, treating the current session as a live sequence and surfacing contextually relevant suggestions as the user shops.

---

*These limitations collectively motivate the project's three-pronged innovation: sequential LSTM modeling (addresses 1, 5), time-aware embeddings (addresses 2), and the three-tier cold start system (addresses 4). Personalization (3) is an emergent property of the LSTM's per-user sequence encoding.*